# NB10 — Refined Markers, Host Species & Annotation Recovery

**Goal**: Fix the over-broad marker panel from Phase 1, extract plant host species,
recover missing annotations via InterProScan domains and KEGG module completeness.

Phase 1 Issues Addressed:
1. Ubiquitous bacterial functions (flagella, chemotaxis, T6SS, biofilm, quorum sensing) inflated dual-nature to 60.3%
2. Plant host species were discarded — only compartment was extracted
3. InterProScan domains never queried — missed T3SS/CWDE Pfam hits
4. KEGG module completeness never checked for multi-gene systems (nif, T3SS)
5. NB04 phylogenetic control failed (missing logit import)

**Outputs**:
- `data/genome_host_species.csv` — genome-level plant host assignments
- `data/species_marker_matrix_v2.csv` — refined marker matrix
- `data/species_cohort_refined.csv` — reclassified cohorts
- `data/interproscan_marker_hits.csv` — InterProScan domain recovery
- `data/kegg_module_completeness.csv` — multi-gene system completeness
- `figures/refined_cohort_comparison.png`

In [1]:
import pandas as pd
import numpy as np
import re, os, warnings
from scipy import stats
warnings.filterwarnings('ignore')

DATA = '../data'
FIG  = '../figures'
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

spark = get_spark_session()
print('Spark session ready')

# Load Phase 1 data
ge = pd.read_csv(f'{DATA}/genome_environment.csv')
sp_comp = pd.read_csv(f'{DATA}/species_compartment.csv')
markers_v1 = pd.read_csv(f'{DATA}/species_marker_matrix.csv')
novel_ogs = pd.read_csv(f'{DATA}/novel_og_annotations.csv')

print(f'Genomes: {len(ge):,}')
print(f'Species: {len(sp_comp):,}')
print(f'Plant-associated species: {sp_comp["is_plant_associated"].sum():,}')
print(f'Novel annotated OGs: {len(novel_ogs)}')

Spark session ready


Genomes: 293,059
Species: 26,511
Plant-associated species: 1,136
Novel annotated OGs: 50


## 1. Plant Host Species Extraction

Parse `ncbi_isolation_source` and `ncbi_env.host` to identify the plant host species
for each genome. Phase 1 extracted only compartment (rhizosphere/root/phyllosphere/endophyte)
but discarded host identity.

In [2]:
# Define plant host patterns
# Map common names and scientific names to standardized host
HOST_PATTERNS = {
    'Oryza sativa': [r'oryza', r'\brice\b'],
    'Triticum aestivum': [r'triticum', r'\bwheat\b'],
    'Zea mays': [r'\bzea\b', r'\bmaize\b', r'\bcorn\b'],
    'Solanum lycopersicum': [r'solanum.*lycopersic', r'\btomato\b'],
    'Solanum tuberosum': [r'solanum.*tuberos', r'\bpotato\b'],
    'Glycine max': [r'glycine.*max', r'\bsoybean\b', r'\bsoja\b'],
    'Arabidopsis thaliana': [r'arabidopsis'],
    'Brassica': [r'brassica', r'\bcabbage\b', r'\bcanola\b', r'\brapeseed\b'],
    'Medicago': [r'medicago', r'\balfalfa\b'],
    'Populus': [r'populus', r'\bpoplar\b'],
    'Citrus': [r'citrus', r'\borange\b', r'\blemon\b', r'\bgrapefruit\b'],
    'Saccharum officinarum': [r'saccharum', r'\bsugarcane\b', r'sugar.*cane'],
    'Gossypium': [r'gossypium', r'\bcotton\b'],
    'Pisum sativum': [r'pisum', r'\bpea\b'],
    'Vitis vinifera': [r'vitis', r'\bgrape\b', r'\bgrapevine\b'],
    'Musa': [r'\bmusa\b', r'\bbanana\b'],
    'Hordeum vulgare': [r'hordeum', r'\bbarley\b'],
    'Lotus japonicus': [r'lotus.*japonic'],
    'Nicotiana': [r'nicotiana', r'\btobacco\b'],
    'Capsicum': [r'capsicum', r'\bpepper\b'],
    'Mangrove': [r'\bmangrove\b'],
    'Sorghum': [r'\bsorghum\b'],
    'Other plant': [],  # catch-all for unmatched plant-associated
}

def extract_host(row):
    """Extract plant host species from isolation_source and host fields."""
    text = ''
    if pd.notna(row.get('ncbi_isolation_source')):
        text += str(row['ncbi_isolation_source']).lower() + ' '
    if pd.notna(row.get('host')):
        text += str(row['host']).lower()
    
    if not text.strip():
        return None
    
    for host_name, patterns in HOST_PATTERNS.items():
        if host_name == 'Other plant':
            continue
        for pat in patterns:
            if re.search(pat, text):
                return host_name
    
    return None

# Apply to plant-associated genomes
plant_genomes = ge[ge['is_plant_associated'] == True].copy()
plant_genomes['host_species'] = plant_genomes.apply(extract_host, axis=1)

# Also check genomes not flagged as plant-associated that have plant hosts
# (they might be associated via host field but not isolation_source)
non_plant = ge[ge['is_plant_associated'] != True].copy()
non_plant['host_species'] = non_plant.apply(extract_host, axis=1)
extra_plant = non_plant[non_plant['host_species'].notna()]
print(f'Additional genomes with plant hosts not flagged as plant-associated: {len(extra_plant)}')

# Combine
all_host = pd.concat([plant_genomes, extra_plant], ignore_index=True)

print(f'\nPlant-associated genomes: {len(plant_genomes)}')
print(f'Genomes with identifiable host: {all_host["host_species"].notna().sum()}')
print(f'\nHost species distribution:')
print(all_host['host_species'].value_counts())

# Save genome-level host assignments
host_cols = ['genome_id', 'gtdb_species_clade_id', 'compartment', 'host_species',
             'ncbi_isolation_source', 'genus', 'phylum']
host_cols = [c for c in host_cols if c in all_host.columns]
all_host[host_cols].to_csv(f'{DATA}/genome_host_species.csv', index=False)
print(f'\nSaved: {DATA}/genome_host_species.csv ({len(all_host)} genomes)')

Additional genomes with plant hosts not flagged as plant-associated: 3857

Plant-associated genomes: 7995
Genomes with identifiable host: 6897

Host species distribution:
host_species
Oryza sativa             1297
Zea mays                  920
Glycine max               828
Solanum lycopersicum      564
Solanum tuberosum         516
Triticum aestivum         423
Arabidopsis thaliana      414
Medicago                  327
Brassica                  237
Saccharum officinarum     202
Citrus                    183
Capsicum                  174
Mangrove                  135
Pisum sativum             125
Vitis vinifera            121
Gossypium                 104
Nicotiana                  87
Populus                    79
Musa                       64
Sorghum                    48
Hordeum vulgare            42
Lotus japonicus             7
Name: count, dtype: int64

Saved: ../data/genome_host_species.csv (11852 genomes)


In [3]:
# Species-level host assignment
# For each species, record all hosts found + dominant host
sp_host = (all_host[all_host['host_species'].notna()]
    .groupby('gtdb_species_clade_id')
    .agg(
        n_host_genomes=('host_species', 'size'),
        dominant_host=('host_species', lambda x: x.value_counts().index[0]),
        n_hosts=('host_species', 'nunique'),
        all_hosts=('host_species', lambda x: ';'.join(sorted(x.unique()))),
    )
    .reset_index())

print(f'Species with identifiable plant host: {len(sp_host)}')
print(f'Species with multiple hosts: {(sp_host["n_hosts"] > 1).sum()}')
print(f'\nDominant host distribution (species-level):')
print(sp_host['dominant_host'].value_counts())

# Which hosts have enough species for H6 (host-specificity) testing?
host_counts = sp_host['dominant_host'].value_counts()
testable = host_counts[host_counts >= 10]
print(f'\nHosts with >= 10 species (testable for H6): {len(testable)}')
print(testable)

Species with identifiable plant host: 1307
Species with multiple hosts: 296

Dominant host distribution (species-level):
dominant_host
Oryza sativa             286
Arabidopsis thaliana     184
Triticum aestivum        122
Zea mays                  98
Solanum tuberosum         77
Mangrove                  75
Glycine max               73
Saccharum officinarum     56
Solanum lycopersicum      55
Brassica                  46
Citrus                    41
Vitis vinifera            37
Populus                   27
Pisum sativum             24
Medicago                  23
Nicotiana                 22
Gossypium                 19
Musa                      13
Capsicum                  12
Sorghum                    8
Hordeum vulgare            5
Lotus japonicus            4
Name: count, dtype: int64

Hosts with >= 10 species (testable for H6): 19
dominant_host
Oryza sativa             286
Arabidopsis thaliana     184
Triticum aestivum        122
Zea mays                  98
Solanum tuberosum      

## 2. Refined Marker Panel

Phase 1 marker panel was too broad — ubiquitous bacterial functions (flagella, chemotaxis,
T6SS, biofilm, quorum sensing) are present in most bacteria regardless of plant association.
This caused 60.3% of species to classify as "dual-nature" (both PGP and pathogenic).

### Refinement strategy:
1. **Remove ubiquitous markers**: flagella, chemotaxis, T6SS, biofilm, quorum sensing, T2SS
2. **Keep plant-specific PGP**: nitrogen fixation, ACC deaminase, phosphate solubilization,
   IAA biosynthesis, DAPG, phenazine, HCN, acetoin/butanediol, siderophore
3. **Keep plant-specific pathogenic**: T3SS, T4SS, effectors, phytotoxin, coronatine, CWDE
4. **Add NB09 novel OGs**: top-scoring novel OGs with plant-interaction hypotheses

In [4]:
# Define refined marker sets
UBIQUITOUS = ['flagella', 'chemotaxis', 't6ss', 'biofilm', 'quorum_sensing', 't2ss']

PGP_MARKERS = ['nitrogen_fixation', 'acc_deaminase', 'phosphate_solubilization',
               'iaa_biosynthesis', 'dapg_biocontrol', 'phenazine', 
               'hydrogen_cyanide', 'acetoin_butanediol', 'siderophore']

PATHOGEN_MARKERS = ['t3ss', 'effector', 'phytotoxin', 'coronatine_toxin',
                    'cwde_cellulase', 'cwde_pectinase', 't4ss',
                    'other_pathogenic']

# Check what markers we have in v1
all_markers_v1 = [c for c in markers_v1.columns if c.endswith('_present')]
marker_base = [c.replace('_present', '') for c in all_markers_v1]

# Markers to keep (present columns)
pgp_keep = [m + '_present' for m in PGP_MARKERS if m + '_present' in markers_v1.columns]
path_keep = [m + '_present' for m in PATHOGEN_MARKERS if m + '_present' in markers_v1.columns]
removed = [m for m in UBIQUITOUS if m + '_present' in markers_v1.columns]

print('Phase 1 markers:', len(all_markers_v1))
print(f'  PGP retained: {len(pgp_keep)} — {", ".join(PGP_MARKERS)}')
print(f'  Pathogen retained: {len(path_keep)} — {", ".join(PATHOGEN_MARKERS)}')
print(f'  Removed (ubiquitous): {len(removed)} — {", ".join(removed)}')

# Show the prevalence of removed markers to justify removal
plant_sp = markers_v1[markers_v1['gtdb_species_clade_id'].isin(
    sp_comp[sp_comp['is_plant_associated'] == True]['gtdb_species_clade_id'])]
nonplant_sp = markers_v1[~markers_v1['gtdb_species_clade_id'].isin(
    sp_comp[sp_comp['is_plant_associated'] == True]['gtdb_species_clade_id'])]

print('\nPrevalence of ubiquitous markers (justifying removal):')
print(f'{"Marker":<20} {"Plant%":>8} {"Non-Plant%":>10} {"Ratio":>7}')
for m in removed:
    col = m + '_present'
    p_prev = plant_sp[col].mean()
    np_prev = nonplant_sp[col].mean()
    ratio = p_prev / max(np_prev, 0.001)
    print(f'{m:<20} {p_prev:>7.1%} {np_prev:>9.1%} {ratio:>7.1f}x')

print('\nPrevalence of retained markers:')
print(f'{"Marker":<25} {"Plant%":>8} {"Non-Plant%":>10} {"Ratio":>7}')
for m in PGP_MARKERS + PATHOGEN_MARKERS:
    col = m + '_present'
    if col in markers_v1.columns:
        p_prev = plant_sp[col].mean()
        np_prev = nonplant_sp[col].mean()
        ratio = p_prev / max(np_prev, 0.001)
        print(f'{m:<25} {p_prev:>7.1%} {np_prev:>9.1%} {ratio:>7.1f}x')

Phase 1 markers: 25
  PGP retained: 9 — nitrogen_fixation, acc_deaminase, phosphate_solubilization, iaa_biosynthesis, dapg_biocontrol, phenazine, hydrogen_cyanide, acetoin_butanediol, siderophore
  Pathogen retained: 8 — t3ss, effector, phytotoxin, coronatine_toxin, cwde_cellulase, cwde_pectinase, t4ss, other_pathogenic
  Removed (ubiquitous): 6 — flagella, chemotaxis, t6ss, biofilm, quorum_sensing, t2ss

Prevalence of ubiquitous markers (justifying removal):
Marker                 Plant% Non-Plant%   Ratio
flagella               68.9%     37.6%     1.8x
chemotaxis             76.1%     49.2%     1.5x
t6ss                   39.7%     16.3%     2.4x
biofilm                30.4%     14.0%     2.2x
quorum_sensing         76.9%     45.3%     1.7x
t2ss                   69.6%     56.5%     1.2x

Prevalence of retained markers:
Marker                      Plant% Non-Plant%   Ratio
nitrogen_fixation           19.6%      8.7%     2.3x
acc_deaminase               15.2%      1.1%    14.4x
phosph

## 3. InterProScan Domain Recovery

Phase 1 used `bakta_annotations` and `bakta_pfam_domains` for marker detection.
InterProScan has additional Pfam hits that bakta may have missed.
Query InterProScan for T3SS, T4SS, CWDE, and siderophore domain families.

In [5]:
CACHE_IPR_MARKERS = f'{DATA}/interproscan_marker_hits.csv'

if os.path.exists(CACHE_IPR_MARKERS) and os.path.getsize(CACHE_IPR_MARKERS) > 100:
    ipr_markers = pd.read_csv(CACHE_IPR_MARKERS)
    print(f'Loaded cached InterProScan marker hits: {len(ipr_markers)} rows')
else:
    # Key Pfam domains for plant-microbe interaction markers
    # T3SS: PF00771 (YscJ/FliF), PF01313 (HrpB), PF01312 (FlhA), PF07201 (SctN)
    # T4SS: PF03743 (VirB8), PF04610 (TrbI), PF07916 (VirB4)
    # CWDE: PF00150 (cellulase), PF01670 (glycoside hydrolase), PF12708 (pectate lyase)
    # Siderophore: PF04183 (IucA/IucC), PF00857 (isochorismatase)
    # Nif: PF00142 (NifH), PF02579 (NifD), PF00148 (NifK)
    # ACC deaminase: PF00266 (AcdS/PLP-dependent)
    
    pfam_ids = [
        # T3SS components
        'PF00771', 'PF01313', 'PF01312', 'PF07201',
        # T4SS components  
        'PF03743', 'PF04610', 'PF07916',
        # Cell wall degrading enzymes
        'PF00150', 'PF01670', 'PF12708', 'PF00295',
        # Siderophore biosynthesis
        'PF04183', 'PF00857',
        # Nitrogen fixation
        'PF00142', 'PF02579', 'PF00148',
        # Effector domains
        'PF09599',  # AvrE family
    ]
    pfam_sql = "','".join(pfam_ids)
    
    ipr_markers = spark.sql(f"""
        SELECT ipr.gene_cluster_id,
               ipr.analysis,
               ipr.signature_acc,
               ipr.signature_desc,
               ipr.ipr_acc,
               ipr.ipr_desc,
               gc.gtdb_species_clade_id,
               gc.is_core
        FROM kbase_ke_pangenome.interproscan_domains ipr
        JOIN kbase_ke_pangenome.gene_cluster gc
             ON ipr.gene_cluster_id = gc.gene_cluster_id
        WHERE ipr.signature_acc IN ('{pfam_sql}')
    """).toPandas()
    
    ipr_markers.to_csv(CACHE_IPR_MARKERS, index=False)
    print(f'Retrieved InterProScan marker hits: {len(ipr_markers)} rows')

print(f'\nInterProScan marker hits by domain:')
print(ipr_markers.groupby(['signature_acc', 'signature_desc'])['gtdb_species_clade_id'].nunique()
      .sort_values(ascending=False).head(20))

print(f'\nTotal species with InterProScan marker domains: {ipr_markers["gtdb_species_clade_id"].nunique()}')

Loaded cached InterProScan marker hits: 280193 rows

InterProScan marker hits by domain:
signature_acc  signature_desc                                               
PF00857        Isochorismatase domain                                           19684
PF01312        FlhB HrpN YscU SpaS Family                                       10225
PF00771        FHIPEP family                                                    10207
PF01313        Bacterial export proteins, family 3                              10088
PF00150        Cellulase (glycosyl hydrolase family 5)                           9722
PF02579        Dinitrogenase iron-molybdenum cofactor                            7549
PF12708        Rhamnogalacturonase A/epimerase, pectate lyase-like               6226
PF00295        Glycosyl hydrolases family 28                                     4890
PF03743        Bacterial conjugation TrbI-like protein                           4733
PF04610        TrbL/VirB6 plasmid conjugal transfer protein 

## 4. KEGG Module Completeness

Phase 1 detected multi-gene systems (nif, T3SS) by finding any single component gene.
A species with just one nifH-like gene but no nifD/K is unlikely to actually fix nitrogen.
Query eggNOG KEGG_Module assignments and check module completeness.

In [6]:
CACHE_KEGG_MOD = f'{DATA}/kegg_module_completeness.csv'

if os.path.exists(CACHE_KEGG_MOD) and os.path.getsize(CACHE_KEGG_MOD) > 100:
    kegg_modules = pd.read_csv(CACHE_KEGG_MOD)
    print(f'Loaded cached KEGG module data: {len(kegg_modules)} rows')
else:
    # Key KEGG modules for plant-microbe interactions
    # M00175: nitrogen fixation (nifH/D/K)
    # M00332: T3SS (multiple components)
    # M00333: T4SS (virB components)
    # M00564: Helicobacter T4SS (cag island)
    # Count distinct gene clusters per module per species as completeness proxy
    kegg_modules = spark.sql("""
        SELECT gc.gtdb_species_clade_id,
               TRIM(km.raw_module) AS module_id,
               COUNT(DISTINCT km.query_name) AS n_clusters
        FROM (
            SELECT query_name,
                   EXPLODE(SPLIT(KEGG_Module, ',')) AS raw_module
            FROM kbase_ke_pangenome.eggnog_mapper_annotations
            WHERE KEGG_Module IS NOT NULL 
              AND KEGG_Module != ''
        ) km
        JOIN kbase_ke_pangenome.gene_cluster gc
             ON km.query_name = gc.gene_cluster_id
        WHERE TRIM(km.raw_module) IN ('M00175', 'M00332', 'M00333', 'M00564')
        GROUP BY gc.gtdb_species_clade_id, TRIM(km.raw_module)
    """).toPandas()
    
    kegg_modules.to_csv(CACHE_KEGG_MOD, index=False)
    print(f'Retrieved KEGG module data: {len(kegg_modules)} species-module pairs')

# Expected minimum gene clusters per module for completeness gating
MODULE_MIN_CLUSTERS = {
    'M00175': 2,   # nifH, nifD, nifK — need at least 2 of 3
    'M00332': 3,   # T3SS has ~10 structural components — need >=3
    'M00333': 3,   # T4SS (VirB1-11) — need >=3
    'M00564': 2,   # Cag T4SS — need >=2
}

kegg_modules['min_clusters'] = kegg_modules['module_id'].map(MODULE_MIN_CLUSTERS)
kegg_modules['passes_gate'] = kegg_modules['n_clusters'] >= kegg_modules['min_clusters']

print('\nKEGG Module Completeness Distribution:')
for mod_id in sorted(kegg_modules['module_id'].unique()):
    sub = kegg_modules[kegg_modules['module_id'] == mod_id]
    min_c = MODULE_MIN_CLUSTERS.get(mod_id, '?')
    print(f'\n{mod_id} (need >= {min_c} gene clusters):')
    print(f'  Species with any hit: {len(sub)}')
    print(f'  Species passing gate: {sub["passes_gate"].sum()}')
    print(f'  Median clusters: {sub["n_clusters"].median():.0f}')
    print(f'  Mean clusters: {sub["n_clusters"].mean():.1f}')

Loaded cached KEGG module data: 19665 rows

KEGG Module Completeness Distribution:

M00175 (need >= 2 gene clusters):
  Species with any hit: 3195
  Species passing gate: 2399
  Median clusters: 3
  Mean clusters: 3.5

M00332 (need >= 3 gene clusters):
  Species with any hit: 5073
  Species passing gate: 1661
  Median clusters: 1
  Mean clusters: 5.1

M00333 (need >= 3 gene clusters):
  Species with any hit: 9081
  Species passing gate: 4515
  Median clusters: 2
  Mean clusters: 7.8

M00564 (need >= 2 gene clusters):
  Species with any hit: 2316
  Species passing gate: 1048
  Median clusters: 1
  Mean clusters: 2.4


## 5. Reclassify Species with Refined Panel

Apply the refined marker panel:
- Remove ubiquitous markers
- Gate nitrogen fixation on KEGG module M00175 >= 50% complete
- Gate T3SS on KEGG module M00332 >= 50% complete (or InterProScan domain recovery)
- Add InterProScan-recovered markers

In [7]:
# Build refined marker matrix
refined = markers_v1[['gtdb_species_clade_id']].copy()

# Copy retained markers
for m in PGP_MARKERS:
    col = m + '_present'
    if col in markers_v1.columns:
        refined[col] = markers_v1[col].values
    else:
        refined[col] = 0

for m in PATHOGEN_MARKERS:
    col = m + '_present'
    if col in markers_v1.columns:
        refined[col] = markers_v1[col].values
    else:
        refined[col] = 0

# Gate nitrogen fixation by KEGG module M00175 cluster count
nif_complete = kegg_modules[kegg_modules['module_id'] == 'M00175']
nif_pass = set(nif_complete[nif_complete['passes_gate'] == True]['gtdb_species_clade_id'])
nif_any = set(refined[refined['nitrogen_fixation_present'] == 1]['gtdb_species_clade_id'])
nif_gated = nif_any - nif_pass
print(f'Nitrogen fixation: {len(nif_any)} species had hits, '
      f'{len(nif_pass)} pass module gate, '
      f'{len(nif_gated)} removed')
refined.loc[refined['gtdb_species_clade_id'].isin(nif_gated), 'nitrogen_fixation_present'] = 0

# Gate T3SS by KEGG module M00332 cluster count
t3ss_complete = kegg_modules[kegg_modules['module_id'] == 'M00332']
t3ss_pass = set(t3ss_complete[t3ss_complete['passes_gate'] == True]['gtdb_species_clade_id'])
t3ss_any = set(refined[refined['t3ss_present'] == 1]['gtdb_species_clade_id'])
t3ss_gated = t3ss_any - t3ss_pass
print(f'T3SS: {len(t3ss_any)} species had hits, '
      f'{len(t3ss_pass)} pass module gate, '
      f'{len(t3ss_gated)} removed')
refined.loc[refined['gtdb_species_clade_id'].isin(t3ss_gated), 't3ss_present'] = 0

# Add InterProScan domain recovery for CWDE (species not already flagged)
cwde_pfams = ['PF00150', 'PF01670', 'PF12708', 'PF00295']
ipr_cwde_species = set(ipr_markers[ipr_markers['signature_acc'].isin(cwde_pfams)]['gtdb_species_clade_id'])
existing_cwde = set(refined[refined['cwde_cellulase_present'] == 1]['gtdb_species_clade_id']) | \
                set(refined[refined['cwde_pectinase_present'] == 1]['gtdb_species_clade_id'])
new_cwde = ipr_cwde_species - existing_cwde
print(f'\nInterProScan CWDE recovery: {len(new_cwde)} additional species')

# Score refined cohorts
pgp_cols = [m + '_present' for m in PGP_MARKERS if m + '_present' in refined.columns]
path_cols = [m + '_present' for m in PATHOGEN_MARKERS if m + '_present' in refined.columns]

refined['n_pgp_refined'] = refined[pgp_cols].sum(axis=1)
refined['n_pathogen_refined'] = refined[path_cols].sum(axis=1)

# Classify
def classify_refined(row):
    pgp = row['n_pgp_refined']
    path = row['n_pathogen_refined']
    if pgp > 0 and path > 0:
        return 'dual-nature'
    elif pgp > 0:
        return 'beneficial'
    elif path > 0:
        return 'pathogenic'
    else:
        return 'neutral'

refined['cohort_refined'] = refined.apply(classify_refined, axis=1)

# Compare v1 vs v2
v1_cohorts = markers_v1['marker_cohort'].value_counts()
v2_cohorts = refined['cohort_refined'].value_counts()

print('\n=== Cohort Comparison: Phase 1 vs Refined ===')
print(f'{"Cohort":<15} {"Phase 1":>10} {"Refined":>10} {"Change":>10}')
for cohort in ['beneficial', 'pathogenic', 'dual-nature', 'neutral']:
    v1 = v1_cohorts.get(cohort, 0)
    v2 = v2_cohorts.get(cohort, 0)
    print(f'{cohort:<15} {v1:>10,} {v2:>10,} {v2-v1:>+10,}')

Nitrogen fixation: 2348 species had hits, 2399 pass module gate, 346 removed
T3SS: 10300 species had hits, 1661 pass module gate, 8855 removed

InterProScan CWDE recovery: 2804 additional species



=== Cohort Comparison: Phase 1 vs Refined ===
Cohort             Phase 1    Refined     Change
beneficial               0      3,394     +3,394
pathogenic               0      6,272     +6,272
dual-nature              0     12,463    +12,463
neutral                971      3,531     +2,560


In [8]:
# Validation: positive and negative controls
# Merge with taxonomy
refined_tax = refined.merge(
    sp_comp[['gtdb_species_clade_id', 'genus', 'phylum', 'is_plant_associated']], 
    on='gtdb_species_clade_id', how='left'
)

# Positive controls: known PGP organisms should be beneficial or dual-nature
pgp_genera = ['g__Rhizobium', 'g__Bradyrhizobium', 'g__Azospirillum', 
              'g__Mesorhizobium', 'g__Sinorhizobium', 'g__Ensifer']
print('=== Positive Controls (known PGP genera) ===')
for g in pgp_genera:
    sub = refined_tax[refined_tax['genus'] == g]
    if len(sub) > 0:
        cohorts = sub['cohort_refined'].value_counts()
        pgp_frac = ((sub['cohort_refined'] == 'beneficial') | 
                     (sub['cohort_refined'] == 'dual-nature')).mean()
        print(f'  {g}: {len(sub)} species | {pgp_frac:.0%} beneficial+dual | {cohorts.to_dict()}')
    else:
        print(f'  {g}: NOT FOUND')

# Negative controls: Escherichia, Salmonella should be neutral or pathogenic (NOT dual-nature/PGP)
neg_genera = ['g__Escherichia', 'g__Salmonella', 'g__Staphylococcus', 'g__Mycobacterium']
print('\n=== Negative Controls (should NOT be PGP) ===')
for g in neg_genera:
    sub = refined_tax[refined_tax['genus'] == g]
    if len(sub) > 0:
        cohorts = sub['cohort_refined'].value_counts()
        pgp_frac = (sub['cohort_refined'] == 'beneficial').mean()
        print(f'  {g}: {len(sub)} species | {pgp_frac:.0%} pure PGP | {cohorts.to_dict()}')

# Plant-associated species breakdown
plant_refined = refined_tax[refined_tax['is_plant_associated'] == True]
print(f'\n=== Plant-associated species cohort distribution ===')
print(plant_refined['cohort_refined'].value_counts())
print(f'\nDual-nature rate: Phase 1 = 60.3%, Refined = {(plant_refined["cohort_refined"]=="dual-nature").mean():.1%}')

=== Positive Controls (known PGP genera) ===
  g__Rhizobium: 63 species | 97% beneficial+dual | {'dual-nature': 61, 'pathogenic': 2}
  g__Bradyrhizobium: 76 species | 100% beneficial+dual | {'dual-nature': 73, 'beneficial': 3}
  g__Azospirillum: 15 species | 100% beneficial+dual | {'dual-nature': 15}
  g__Mesorhizobium: 69 species | 99% beneficial+dual | {'dual-nature': 67, 'pathogenic': 1, 'beneficial': 1}
  g__Sinorhizobium: 11 species | 100% beneficial+dual | {'dual-nature': 11}
  g__Ensifer: 9 species | 100% beneficial+dual | {'dual-nature': 9}

=== Negative Controls (should NOT be PGP) ===
  g__Escherichia: 8 species | 25% pure PGP | {'dual-nature': 6, 'beneficial': 2}
  g__Salmonella: 5 species | 0% pure PGP | {'dual-nature': 5}
  g__Staphylococcus: 61 species | 18% pure PGP | {'dual-nature': 49, 'beneficial': 11, 'pathogenic': 1}
  g__Mycobacterium: 139 species | 6% pure PGP | {'dual-nature': 102, 'pathogenic': 24, 'beneficial': 8, 'neutral': 5}

=== Plant-associated species coh

## 6. Phylogenetic Control (fixing NB04 failure)

NB04 phylogenetic control failed due to missing statsmodels.api import.
Run logistic regression with genus-level fixed effects for key markers.

In [9]:
from statsmodels.discrete.discrete_model import Logit
from statsmodels.tools import add_constant
from statsmodels.stats.multitest import multipletests

# Test: does plant association predict each marker after controlling for genus?
# Use top-20 most common genera as fixed effects (full genus effects = too many params)
top_genera = refined_tax['genus'].value_counts().head(20).index.tolist()

# Create genus dummies for top 20
genus_dummies = pd.get_dummies(refined_tax['genus'].where(
    refined_tax['genus'].isin(top_genera), 'other'
), prefix='genus', drop_first=True)

results = []
for marker in PGP_MARKERS + PATHOGEN_MARKERS:
    col = marker + '_present'
    if col not in refined_tax.columns:
        continue
    if refined_tax[col].sum() < 50:
        continue
    
    y = refined_tax[col].values
    X = pd.concat([
        refined_tax[['is_plant_associated']].fillna(False).astype(int),
        genus_dummies
    ], axis=1)
    X = add_constant(X)
    
    try:
        model = Logit(y, X).fit(disp=0, maxiter=100)
        coef = model.params['is_plant_associated']
        pval = model.pvalues['is_plant_associated']
        or_val = np.exp(coef)
        results.append({
            'marker': marker,
            'coefficient': coef,
            'odds_ratio': or_val,
            'p_value': pval,
            'n_positive': int(y.sum()),
            'converged': model.mle_retvals['converged'],
        })
    except Exception as e:
        results.append({
            'marker': marker,
            'coefficient': np.nan,
            'odds_ratio': np.nan,
            'p_value': np.nan,
            'n_positive': int(y.sum()),
            'converged': False,
        })

phylo_results = pd.DataFrame(results)

# BH-FDR correction
valid = phylo_results['p_value'].notna()
if valid.sum() > 0:
    _, q_vals, _, _ = multipletests(phylo_results.loc[valid, 'p_value'], method='fdr_bh')
    phylo_results.loc[valid, 'q_value'] = q_vals

print('=== Phylogenetic Control: Plant Association -> Marker (Genus Fixed Effects) ===')
print(f'{"Marker":<25} {"OR":>6} {"p-value":>10} {"q-value":>10} {"n+":>6} {"Conv":>5}')
for _, row in phylo_results.sort_values('p_value').iterrows():
    sig = '*' if row.get('q_value', 1) < 0.05 else ''
    conv = 'Y' if row['converged'] else 'N'
    print(f'{row["marker"]:<25} {row["odds_ratio"]:>6.2f} {row["p_value"]:>10.2e} '
          f'{row.get("q_value", np.nan):>10.2e} {row["n_positive"]:>6} {conv:>5} {sig}')

=== Phylogenetic Control: Plant Association -> Marker (Genus Fixed Effects) ===
Marker                        OR    p-value    q-value     n+  Conv
nitrogen_fixation            nan        nan        nan   2002     N 
acc_deaminase                nan        nan        nan    430     N 
phosphate_solubilization     nan        nan        nan   9415     N 
iaa_biosynthesis             nan        nan        nan    214     N 
dapg_biocontrol              nan        nan        nan    118     N 
phenazine                    nan        nan        nan   7818     N 
hydrogen_cyanide             nan        nan        nan   1123     N 
acetoin_butanediol           nan        nan        nan   1876     N 
siderophore                  nan        nan        nan   1495     N 
t3ss                         nan        nan        nan   1445     N 
effector                     nan        nan        nan   5589     N 
cwde_cellulase               nan        nan        nan  12265     N 
cwde_pectinase          

In [10]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Phase 1 vs Refined cohort distribution
cohorts = ['beneficial', 'pathogenic', 'dual-nature', 'neutral']
v1_vals = [v1_cohorts.get(c, 0) for c in cohorts]
v2_vals = [v2_cohorts.get(c, 0) for c in cohorts]

x = np.arange(len(cohorts))
w = 0.35
axes[0].bar(x - w/2, v1_vals, w, label='Phase 1', color='lightcoral', edgecolor='black')
axes[0].bar(x + w/2, v2_vals, w, label='Refined', color='steelblue', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cohorts, rotation=30, ha='right')
axes[0].set_ylabel('Number of species')
axes[0].set_title('A) Cohort Assignment: Phase 1 vs Refined')
axes[0].legend()

# Panel B: Marker prevalence in plant vs non-plant (refined set only)
marker_data = []
for m in PGP_MARKERS + PATHOGEN_MARKERS:
    col = m + '_present'
    if col in refined_tax.columns:
        p_prev = refined_tax.loc[refined_tax['is_plant_associated'] == True, col].mean()
        np_prev = refined_tax.loc[refined_tax['is_plant_associated'] != True, col].mean()
        marker_data.append({'marker': m, 'plant': p_prev, 'non_plant': np_prev})

mdf = pd.DataFrame(marker_data)
y_pos = np.arange(len(mdf))
axes[1].barh(y_pos - 0.15, mdf['plant'], 0.3, label='Plant-associated', color='forestgreen', alpha=0.7)
axes[1].barh(y_pos + 0.15, mdf['non_plant'], 0.3, label='Non-plant', color='gray', alpha=0.7)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(mdf['marker'], fontsize=8)
axes[1].set_xlabel('Prevalence')
axes[1].set_title('B) Marker Prevalence (Refined Panel)')
axes[1].legend(fontsize=8)
axes[1].invert_yaxis()

# Panel C: Phylogenetic control OR
pr = phylo_results.dropna(subset=['odds_ratio']).sort_values('odds_ratio', ascending=True)
colors = ['forestgreen' if m in PGP_MARKERS else 'tomato' for m in pr['marker']]
axes[2].barh(range(len(pr)), pr['odds_ratio'], color=colors, edgecolor='black', alpha=0.7)
axes[2].set_yticks(range(len(pr)))
axes[2].set_yticklabels(pr['marker'], fontsize=8)
axes[2].axvline(x=1.0, color='black', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Odds Ratio (plant vs non-plant, genus-controlled)')
axes[2].set_title('C) Phylogenetic Control (Genus Fixed Effects)')
# Add significance markers
for i, (_, row) in enumerate(pr.iterrows()):
    if row.get('q_value', 1) < 0.05:
        axes[2].text(row['odds_ratio'] + 0.05, i, '*', fontsize=12, va='center')

plt.tight_layout()
plt.savefig(f'{FIG}/refined_cohort_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG}/refined_cohort_comparison.png')

Saved: ../figures/refined_cohort_comparison.png


In [11]:
# Save all outputs
refined.to_csv(f'{DATA}/species_marker_matrix_v2.csv', index=False)
print(f'Saved: {DATA}/species_marker_matrix_v2.csv ({len(refined)} species, '
      f'{len(pgp_cols) + len(path_cols)} markers)')

# Save refined cohort assignments with taxonomy
cohort_out = refined_tax[['gtdb_species_clade_id', 'genus', 'phylum', 'is_plant_associated',
                           'cohort_refined', 'n_pgp_refined', 'n_pathogen_refined']].copy()
cohort_out = cohort_out.merge(sp_host, on='gtdb_species_clade_id', how='left')
cohort_out.to_csv(f'{DATA}/species_cohort_refined.csv', index=False)
print(f'Saved: {DATA}/species_cohort_refined.csv')

# Save phylogenetic control results
phylo_results.to_csv(f'{DATA}/phylo_control_results.csv', index=False)
print(f'Saved: {DATA}/phylo_control_results.csv')

Saved: ../data/species_marker_matrix_v2.csv (25660 species, 17 markers)
Saved: ../data/species_cohort_refined.csv
Saved: ../data/phylo_control_results.csv


In [12]:
# Final summary
print('\n' + '='*80)
print('NB10 SUMMARY: Refined Markers, Host Species & Annotation Recovery')
print('='*80)

print(f'\n1. Host Species Extraction:')
print(f'   Genomes with plant host: {all_host["host_species"].notna().sum():,}')
print(f'   Species with plant host: {len(sp_host)}')
print(f'   Testable hosts (>= 10 species): {len(testable)}')

print(f'\n2. Marker Panel Refinement:')
print(f'   Markers removed (ubiquitous): {len(removed)} — {", ".join(removed)}')
print(f'   PGP markers retained: {len(pgp_cols)}')
print(f'   Pathogen markers retained: {len(path_cols)}')
print(f'   Nitrogen fixation gated: {len(nif_gated)} species lost nif status')
print(f'   T3SS gated: {len(t3ss_gated)} species lost T3SS status')

print(f'\n3. Cohort Changes:')
for cohort in ['beneficial', 'pathogenic', 'dual-nature', 'neutral']:
    v1 = v1_cohorts.get(cohort, 0)
    v2 = v2_cohorts.get(cohort, 0)
    print(f'   {cohort}: {v1:,} -> {v2:,} ({v2-v1:+,})')

print(f'\n4. InterProScan Recovery:')
print(f'   Total marker domain hits: {len(ipr_markers):,}')
print(f'   Species with marker domains: {ipr_markers["gtdb_species_clade_id"].nunique():,}')
print(f'   New CWDE species from InterProScan: {len(new_cwde)}')

sig_markers = phylo_results[phylo_results['q_value'] < 0.05] if 'q_value' in phylo_results.columns else pd.DataFrame()
print(f'\n5. Phylogenetic Control:')
print(f'   Markers tested: {len(phylo_results)}')
print(f'   Significant after genus control (FDR < 0.05): {len(sig_markers)}')

print(f'\nOutputs:')
print(f'  data/genome_host_species.csv')
print(f'  data/species_marker_matrix_v2.csv')
print(f'  data/species_cohort_refined.csv')
print(f'  data/interproscan_marker_hits.csv')
print(f'  data/kegg_module_completeness.csv')
print(f'  data/phylo_control_results.csv')
print(f'  figures/refined_cohort_comparison.png')


NB10 SUMMARY: Refined Markers, Host Species & Annotation Recovery

1. Host Species Extraction:
   Genomes with plant host: 6,897
   Species with plant host: 1307
   Testable hosts (>= 10 species): 19

2. Marker Panel Refinement:
   Markers removed (ubiquitous): 6 — flagella, chemotaxis, t6ss, biofilm, quorum_sensing, t2ss
   PGP markers retained: 9
   Pathogen markers retained: 8
   Nitrogen fixation gated: 346 species lost nif status
   T3SS gated: 8855 species lost T3SS status

3. Cohort Changes:
   beneficial: 0 -> 3,394 (+3,394)
   pathogenic: 0 -> 6,272 (+6,272)
   dual-nature: 0 -> 12,463 (+12,463)
   neutral: 971 -> 3,531 (+2,560)

4. InterProScan Recovery:
   Total marker domain hits: 280,193
   Species with marker domains: 24,595
   New CWDE species from InterProScan: 2804

5. Phylogenetic Control:
   Markers tested: 14
   Significant after genus control (FDR < 0.05): 0

Outputs:
  data/genome_host_species.csv
  data/species_marker_matrix_v2.csv
  data/species_cohort_refined.